# Imports

In [83]:
import os
import json
import yaml
from pathlib import Path
import re
import datetime


import boto3
from dotenv import load_dotenv

# Environmental Variables

In [105]:
# set S3_BUCKET environmental variable to arn:aws:s3:::kasbench-test-20260528-377288663341-us-east-1-an
os.environ["S3_BUCKET"] = ""


In [18]:
load_dotenv()
s3_bucket = os.environ.get("S3_BUCKET")
print(f"S3 Bucket: {s3_bucket}")



S3 Bucket: kasbench-test-20260528-377288663341-us-east-1-an


# Constants

In [34]:
run_id = "verification-15-percentile-large-a"
s3 = boto3.client("s3")

data_directory = "../data"



monitored_events_text = """NodeNotReady
NodeUnreachable
NodeHasMemoryPressure
NodeHasDiskPressure
NodeHasPIDPressure
Evicted
OOMKilling
SystemOOM
FailedMount
FailedAttachVolume
FailedCreatePodSandBox
FailedPull
ErrImagePull
ImagePullBackOff
NetworkNotReady
FailedScheduling
Unhealthy
BackOff
CrashLoopBackOff
FailedCreate
FailedKillPod
FailedSync
Rebooted"""

monitored_events = [event for event in monitored_events_text.split("\n") if event]


# Functions

In [ ]:
def download_file_from_s3(s3_bucket, s3_key, local_path):
    """Download a file from S3 to a local path."""
    s3.download_file(s3_bucket, s3_key, local_path)
    print(f"Downloaded {s3_key} to {local_path}")


def get_s3_file_listing(s3_bucket, s3_prefix=''):
    """Get a listing of files in an S3 bucket."""
    file_list = []
    paginator = s3.get_paginator('list_objects_v2')
    pages = paginator.paginate(Bucket=s3_bucket, Prefix=s3_prefix)
    for page in pages:
        if 'Contents' in page:
            for obj in page['Contents']:
                file_list.append(obj)
    return file_list

def get_s3_file(s3_bucket, s3_filename, local_filename, data_directory, force=False):
    """Download a file from S3 to a local path."""

    # rename the local file
    local_filename = f"{data_directory}/{local_filename}"

    # skip if file exists and not force
    if os.path.exists(local_filename) and not force: 
        return local_filename
    
    
    # create the local directory, if it doesn't exist 

    dir_path = str(Path(f"{data_directory}/{local_filename}").parent)
    os.makedirs(dir_path, exist_ok=True)

    # download the tile
    s3.download_file(s3_bucket, s3_filename, local_filename)
    return local_filename

def parse_trial(filename):
    pattern = r"^[^/]+/(trial\d{4})/"
    if match := re.match(pattern, filename):
        return match.group(1)
    return None

In [88]:
def get_trial_start_end_times(files, run_id, s3_bucket, data_directory, force=False, verbose=False):
    trials = {}
    for file in files:
        if trial_id := parse_trial((file["Key"])):
            s3_filename = f"{run_id}/{trial_id}/run_details.json"
            local_filename = get_s3_file(s3_bucket, s3_filename, s3_filename, data_directory, force=force)
            with open(local_filename, "r") as f:
                trial = json.load(f)
                start_time = trial["status"]["startTime"] # string in format '2026-09-18T21:18:54.489000+00:00'
                end_time = trial["status"]["endTime"] # string in format '2026-09-18T21:18:54.489000+00:00'
                # convert start_time and end_time to timezone-aware dateteime objects
                start_time = datetime.datetime.fromisoformat(start_time.replace('Z', '+00:00'))
                end_time = datetime.datetime.fromisoformat(end_time.replace('Z', '+00:00'))
                trials[trial_id] = {"start_time": start_time, "end_time": end_time}

    if verbose:
        for trial_id, times in trials.items():
            print(f"{trial_id}: {times['start_time']} -> {times['end_time']}")

    return trials
                

# Code

## Get the listing of files in the S3 bucket for the specified run

In [86]:
files = get_s3_file_listing(s3_bucket, run_id)


## Get the starting and ending time of each trial

In [89]:
trial_times = get_trial_start_end_times(files, run_id, s3_bucket, data_directory, force=False, verbose=False)        
trial_times

{'trial0001': {'start_time': datetime.datetime(2026, 9, 18, 20, 48, 48, 765649, tzinfo=datetime.timezone.utc),
  'end_time': datetime.datetime(2026, 9, 18, 21, 18, 54, 489000, tzinfo=datetime.timezone.utc)},
 'trial0002': {'start_time': datetime.datetime(2026, 9, 18, 21, 40, 44, 31588, tzinfo=datetime.timezone.utc),
  'end_time': datetime.datetime(2026, 9, 18, 22, 10, 49, 394000, tzinfo=datetime.timezone.utc)},
 'trial0003': {'start_time': datetime.datetime(2026, 9, 18, 22, 33, 0, 262451, tzinfo=datetime.timezone.utc),
  'end_time': datetime.datetime(2026, 9, 18, 23, 3, 5, 59000, tzinfo=datetime.timezone.utc)},
 'trial0004': {'start_time': datetime.datetime(2026, 9, 18, 23, 24, 46, 327266, tzinfo=datetime.timezone.utc),
  'end_time': datetime.datetime(2026, 9, 18, 23, 54, 50, 591000, tzinfo=datetime.timezone.utc)},
 'trial0005': {'start_time': datetime.datetime(2026, 9, 19, 0, 15, 53, 829765, tzinfo=datetime.timezone.utc),
  'end_time': datetime.datetime(2026, 9, 19, 0, 45, 59, 883000,

## Analyze event logs (warnings.yaml)

In [104]:
types = set()
results = []
        
for file in files:
    if "post/events/all.yaml" in file["Key"]:
        filename = file["Key"]
        
        # parse the trial id from the filename
        trial_id = parse_trial(filename)
        
        # download the event file
        filename = get_s3_file(s3_bucket, filename, filename, data_directory)
        
        # load the event file
        with open(filename, "r") as f:
            events = yaml.load(f, Loader=yaml.FullLoader)
            
        counter = 0
        for item in events["items"]:
            reason = item["reason"]
            type = item["type"]
            types.add(type)        
            if reason in monitored_events:
                
                first_timestamp = item["firstTimestamp"] # string in format 2026-09-18T20:39:54Z
                last_timestamp = item["lastTimestamp"] # string in format 2026-09-18T20:39:54Z
                
                # convert first_timestamp and last_timestamp to timezone aware datetimes
                first_timestamp = datetime.datetime.fromisoformat(first_timestamp.replace('Z', '+00:00'))
                last_timestamp = datetime.datetime.fromisoformat(last_timestamp.replace('Z', '+00:00'))
                
                # exclude events before and after the trial run
                if last_timestamp < trial_times[trial_id]["start_time"] or first_timestamp > trial_times[trial_id]["end_time"]:
                    continue

                # gather relevant fields
                name = item["metadata"]["name"]
                namespace = item["metadata"]["namespace"]
                message = item["message"]
                type = item["type"]

                # exclude probe failures in the globeco namespace
                if namespace == "globeco" and re.search("^(Readiness|Startup|Liveness) probe failed", message):
                    continue
                

                counter += 1
                print(f"*** {type}: {reason} ***")
                print(f"{first_timestamp} {last_timestamp}")
                print(f"{name} {namespace}")
                print(message)
                print()

                results.append({
                    "type": type,
                    "reason": reason,
                    "first_timestamp": first_timestamp,
                    "last_timestamp": last_timestamp,
                    "name": name,
                    "namespace": namespace,
                    "message": message,
                    "trial_id": trial_id
                })
                
        # print(f"Trial: {trial_id} \tCounter: {counter}")    
        

print(types)    
print(results)

{'Normal', 'Warning'}
[]


In [103]:
results

[]